# Aula 04 — BM25: o modelo probabilístico, construído do zero (Teoria)
**Projeto Integrador III — Motor de Busca**

**Alunos:** Ivan Guimarães dos S. da Silva, Jorge Sugano Suzart e Gustavo Geminiano

*Kernel: R | Google Colaboratory (IRkernel)*

In [1]:
# 1. Setup oficial do motor04 e inspecao do estado herdado
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor04.R")
docs <- docs_aula()
cfg  <- cfg_aula()
ix   <- montar(docs, cfg)
estado()

[1] "MOTOR_VERSAO: motor04 v1 2026-10-01"
| objeto | tipo | tamanho | valor / amostra |
|---|---|---|---|
| `cfg` | list | 3 | elementos: limpar, acentos, stopwords |
| `docs` | character, nomeado | 8 | d1 recuperacao de informacao ordena..., d2 o modelo de espaco vetorial repre..., ... |
| `ix` | list | 9 | elementos: tokens, vocab, tf, N, df, idf_tfidf, w, wn, postings |


## Módulo 1 — Por que ir além do TF-IDF
Demonstração dos dois defeitos estruturais do modelo vetorial TF-IDF clássico: crescimento linear da frequência sem saturação e penalização/acumulação inadequada de tamanho.

In [2]:
# Módulo 1: O ranking do cosseno da Aula 02 como ponto de partida
cat("Ranking do cosseno para 'modelo de recuperacao':\n")
print(round(ranking_cosseno("modelo de recuperacao", ix, cfg), 3))

cat("\nPeso TF-IDF de 'de' em d1 (1x) vs d3 (2x):\n")
cat("d1: ", ix$w["de", "d1"], "| d3: ", ix$w["de", "d3"], "\n")

Ranking do cosseno para 'modelo de recuperacao':
   d1    d3    d4    d2    d6    d8    d5    d7 
0.254 0.233 0.215 0.208 0.025 0.023 0.000 0.000 

Peso TF-IDF de 'de' em d1 (1x) vs d3 (2x):
d1:  0.4700036 | d3:  0.9400073 


## Módulo 2 — Poisson: a distribuição e os números
Modelagem probabilística de eventos raros via distribuição de Poisson $P(X = f) = \frac{e^{-\lambda} \lambda^f}{f!}$.

In [3]:
# Módulo 2: Distribuicao de Poisson para contagem de termos
f <- 0:5
p_lam2 <- exp(-2) * 2^f / factorial(f)
cat("P(X = f) com lambda = 2 (f = 0 a 5):\n")
print(round(p_lam2, 3))

# Checkpoint 2: lambda = 3
p_lam3 <- exp(-3) * 3^c(0, 3, 6) / factorial(c(0, 3, 6))
cat("\nCheckpoint 2 (lambda = 3 para f = 0, 3 e 6):\n")
print(round(p_lam3, 3))

P(X = f) com lambda = 2 (f = 0 a 5):
[1] 0.135 0.271 0.271 0.180 0.090 0.036

Checkpoint 2 (lambda = 3 para f = 0, 3 e 6):
[1] 0.050 0.224 0.050


## Módulo 3 — Burstiness: quanto uma Poisson só erra
O fenômeno de rajadas textuais (*burstiness*) e a razão de falha ($216\times$ em $f=6$) de uma distribuição de Poisson unimodal homogênea.

In [4]:
# Módulo 3: Mistura 2-Poisson vs Poisson unica
# Mistura: 15% lambda=4, 85% mu=0.1. Media ponderada = 0.685
f_burst <- c(0, 2, 4, 6)
p_mistura <- 0.15 * (exp(-4) * 4^f_burst / factorial(f_burst)) + 
             0.85 * (exp(-0.1) * 0.1^f_burst / factorial(f_burst))

med_global <- 0.15 * 4 + 0.85 * 0.1
p_unica <- exp(-med_global) * med_global^f_burst / factorial(f_burst)

razao <- p_mistura / p_unica
tabela_burst <- data.frame(f = f_burst, Mistura = round(p_mistura, 5), 
                           Poisson_Unica = round(p_unica, 5), Razao = round(razao, 1))
print(tabela_burst)

  f Mistura Poisson_Unica Razao
1 0 0.77196       0.50409   1.5
2 2 0.02568       0.11827   0.2
3 4 0.02928       0.00463   6.3
4 6 0.01560       0.00007 216.0


## Módulos 4 e 5 — Eliteness e a Saturação 2-Poisson
Definição das populações latentes Elite e Não-Elite e dedução analítica da saturação assintótica com normalização $w(0) = -0{,}7304$ e teto $2{,}5221$.

In [5]:
# Módulos 4 e 5: Curva de peso 2-Poisson normalizada
w2p <- function(f, p = 0.6, q = 0.1, lam = 3, mu = 0.2) {
  num <- p * exp(-lam) * lam^f + (1 - p) * exp(-mu) * mu^f
  den <- q * exp(-lam) * lam^f + (1 - q) * exp(-mu) * mu^f
  log(num / den)
}

cat("w(0) cru (penalidade de ausencia): ", round(w2p(0), 4), "\n")
cat("Teto cru w(infinito) = log(p/q)    : ", round(log(0.6 / 0.1), 4), "\n")
cat("Teto normalizado w(inf) - w(0)    : ", round(log(0.6 / 0.1) - w2p(0), 4), "\n\n")

cat("Curva normalizada para f = 0, 1, 2, 3, 5, 8:\n")
print(round(w2p(c(0, 1, 2, 3, 5, 8)) - w2p(0), 3))

# Checkpoint 5: com q = 0.2
cat("\nCheckpoint 5 (com q = 0.2):\n")
cat("Teto normalizado: ", round(log(0.6 / 0.2) - w2p(0, q = 0.2), 4), "\n")

w(0) cru (penalidade de ausencia):  -0.7304 
Teto cru w(infinito) = log(p/q)    :  1.7918 
Teto normalizado w(inf) - w(0)    :  2.5221 

Curva normalizada para f = 0, 1, 2, 3, 5, 8:
[1] 0.000 0.685 2.064 2.482 2.522 2.522

Checkpoint 5 (com q = 0.2):
Teto normalizado:  1.7196 


## Módulo 6 — Aproximação Hiperbólica de Robertson
A substituição prática dos 4 parâmetros latentes inobserváveis pela hipérbole $w(f) \approx \text{IDF}(t) \cdot \frac{f(k_1+1)}{f+k_1}$.

In [6]:
# Módulo 6: Curva hiperbolica de saturacao vs TF linear
sat <- function(f, k1 = 1.2) (f * (k1 + 1)) / (f + k1)

cat("Saturacao hiperbolica (k1 = 1.2) para f = 0 a 5:\n")
print(round(sat(0:5), 3))

# Checkpoint 6: Comparacao de candidatas
f_cand <- 0:3
cand_exp  <- 1 - 0.5^f_cand
cand_sqrt <- sqrt(f_cand)
cat("\nCandidata 1 - 0.5^f : ", round(cand_exp, 3), "(passa nas 4 propriedades)\n")
cat("Candidata sqrt(f)   : ", round(cand_sqrt, 3), "(falha na saturacao/sem teto)\n")

Saturacao hiperbolica (k1 = 1.2) para f = 0 a 5:
[1] 0.000 1.000 1.375 1.571 1.692 1.774

Candidata 1 - 0.5^f :  0 0.5 0.75 0.875 (passa nas 4 propriedades)
Candidata sqrt(f)   :  0 1 1.414 1.732 (falha na saturacao/sem teto)


## Módulo 7 — A fórmula do BM25: k1, b e K
Incorporação da normalização por comprimento com interpolação $b=0{,}75$ entre verbosidade e escopo.

In [7]:
# Módulo 7: Penalizacao de tamanho de documento K
dl <- colSums(ix$tf)
avgdl <- mean(dl)
k1 <- 1.2
b  <- 0.75

K <- k1 * (1 - b + b * dl / avgdl)
contrib_f1 <- (k1 + 1) / (1 + K)

tab_k <- data.frame(doc = names(dl), dl = dl, dl_sobre_avgdl = dl / avgdl, 
                    K = round(K, 3), contrib = round(contrib_f1, 3))
print(tab_k[c("d4", "d1", "d6", "d3"), ])

# Checkpoint 7: documento com |d| = 16
K_16 <- k1 * (1 - b + b * 16 / 8)
cat("\nCheckpoint 7 (|d| = 16, avgdl = 8):\n")
cat("K: ", K_16, "| Contribuicao: ", round((k1 + 1) / (1 + K_16), 3), "\n")

   doc dl dl_sobre_avgdl     K contrib
d4  d4  6          0.750 0.975   1.114
d1  d1  7          0.875 1.087   1.054
d6  d6  8          1.000 1.200   1.000
d3  d3  9          1.125 1.312   0.951

Checkpoint 7 (|d| = 16, avgdl = 8):
K:  2.1 | Contribuicao:  0.71 


## Módulo 8 — O IDF do BM25
Cálculo da razão de chances suavizada (RSJ com $+0{,}5$) e a importância do $+1$ contra pesos negativos.

In [8]:
# Módulo 8: Calculo do IDF do BM25
idf_b <- log((ix$N - ix$df + 0.5) / (ix$df + 0.5) + 1)

cat("IDFs do BM25 para os termos da consulta:\n")
print(round(idf_b[c("modelo", "recuperacao", "documentos", "de")], 3))

# Checkpoint 8: termo em 7 de 8 documentos
df_test <- 7
idf_c8        <- log((8 - df_test + 0.5) / (df_test + 0.5) + 1)
idf_sem_mais1 <- log((8 - df_test + 0.5) / (df_test + 0.5))
idf_sem_meio  <- log(8 / df_test)

cat("\nCheckpoint 8 (df = 7, N = 8):\n")
cat("BM25 oficial       : ", round(idf_c8, 3), "\n")
cat("Sem o +1 (negativo): ", round(idf_sem_mais1, 3), "\n")
cat("Sem o +0.5 (Aula 01): ", round(idf_sem_meio, 3), "\n")

IDFs do BM25 para os termos da consulta:
     modelo recuperacao  documentos          de 
      1.281       1.281       0.693       0.492 

Checkpoint 8 (df = 7, N = 8):
BM25 oficial       :  0.182 
Sem o +1 (negativo):  -1.609 
Sem o +0.5 (Aula 01):  0.134 


## Módulo 9 — Implementação Completa do BM25 em R
Construção da função canônica `bm25_doc()` com proteção contra `NaN` e análise comparativa de rankings e parâmetros.

In [9]:
# Módulo 9: Implementacao canônica do BM25 e ranking
bm25_doc <- function(termos, d, k1 = 1.2, b = 0.75) {
  s <- 0
  for (t in termos) {
    if (!t %in% ix$vocab) next
    f <- ix$tf[t, d]
    if (f == 0) next
    K <- k1 * (1 - b + b * dl[[d]] / avgdl)
    s <- s + idf_b[[t]] * (f * (k1 + 1)) / (f + K)
  }
  s
}

consulta <- preparar("modelo de recuperacao", cfg)
scores <- sapply(colnames(ix$tf), function(d) bm25_doc(consulta, d))

cat("Ranking oficial BM25 para 'modelo de recuperacao':\n")
print(round(sort(scores, decreasing = TRUE), 3))

# Checkpoint 9: Calculo analitico de d2
score_d2_manual <- (idf_b[["modelo"]] + idf_b[["de"]]) * ((k1 + 1) / (1 + K[["d2"]]))
cat("\nCheckpoint 9 - d2 calculado analiticamente: ", round(score_d2_manual, 3), "\n")

# Variacoes de parametros
s_k05 <- sapply(colnames(ix$tf), function(d) bm25_doc(consulta, d, k1 = 0.5))
cat("\nVariacao k1 = 0.5 (d1 ultrapassa d3):\n")
print(round(sort(s_k05, decreasing = TRUE), 3))

s_b0 <- sapply(colnames(ix$tf), function(d) bm25_doc(consulta, d, b = 0))
cat("\nVariacao b = 0 (tamanho ignorado):\n")
print(round(sort(s_b0, decreasing = TRUE), 3))

Ranking oficial BM25 para 'modelo de recuperacao':
   d3    d1    d2    d4    d8    d6    d5    d7 
1.873 1.869 1.687 1.427 0.519 0.492 0.000 0.000 

Checkpoint 9 - d2 calculado analiticamente:  1.687 

Variacao k1 = 0.5 (d1 ultrapassa d3):
   d1    d3    d2    d4    d8    d6    d5    d7 
1.831 1.822 1.720 1.366 0.508 0.492 0.000 0.000 

Variacao b = 0 (tamanho ignorado):
   d3    d1    d2    d4    d6    d8    d5    d7 
1.958 1.773 1.773 1.281 0.492 0.492 0.000 0.000 


## Fechamento da Sessão e Snapshot de Estado
Anexação da memória do R no consolidado.

In [10]:
# Fechamento e persistencia de estado
if (file.exists("aula04_consolidado.md")) {
  anexar_estado("aula04_consolidado.md")
  cat("Estado anexado a aula04_consolidado.md com sucesso!\n")
}
cat("Sessao teorica da Aula 04 finalizada!\n")

Estado anexado a aula04_consolidado.md com sucesso!
Sessao teorica da Aula 04 finalizada!
